In [ ]:
import torch
import sentencepiece as spm
from mytransformer.model import Transformer
from mytransformer.masks import make_pad_mask
from mytransformer.decode import beam_search_decode

# 1. Setup Device and Tokenizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sp = spm.SentencePieceProcessor(model_file='m30k_bpe.model')

VOCAB_SIZE = sp.get_piece_size()
PAD_IDX = sp.pad_id()
BOS_IDX = sp.bos_id()
EOS_IDX = sp.eos_id()

# 2. Load the Trained Model
model = Transformer(
    src_vocab=VOCAB_SIZE, tgt_vocab=VOCAB_SIZE, 
    d_model=256, heads=4, num_layers=3, d_ff=1024, dropout=0.1
).to(device)

model.load_state_dict(torch.load("translation_model.pt", map_location=device))
model.eval()
print("Model loaded successfully!")

# 3. Translation Helper Function
def translate(german_sentence):
    src_encoded = [BOS_IDX] + sp.encode_as_ids(german_sentence) + [EOS_IDX]
    src_tensor = torch.tensor(src_encoded).unsqueeze(0).to(device)
    src_mask = make_pad_mask(src_tensor, PAD_IDX).to(device)
    
    output_ids = beam_search_decode(model, src_tensor, src_mask, max_len=50, bos_idx=BOS_IDX, eos_idx=EOS_IDX, device=device)
    
    out_list = output_ids.squeeze().tolist()
    if isinstance(out_list, int): out_list = [out_list]
    clean_out = [idx for idx in out_list if idx not in (BOS_IDX, EOS_IDX, PAD_IDX)]
    
    return sp.decode_ids(clean_out)

# 4. Test Interactive Translations
test_sentences = [
    "Der Morgen beginnt meistens sehr ruhig, wenn die ersten Sonnenstrahlen durch das Fenster scheinen. Viele Menschen trinken eine Tische Kaffee oder Tee, bevor sie sich auf den Weg zur Arbeit machen. Auf den Straßen herrscht oft schon große Hektik, während die Autos im dichten Verkehr stehen. Doch wenn man am Nachmittag einen Spaziergang im Park macht, kann man die frische Luft genießen und den Gesang der Vögel hören. Die Natur bietet uns einen wunderbaren Ausgleich zum stressigen Alltag in der modernen Stadt."

]

for sentence in test_sentences:
    print(f"DE: {sentence}")
    print(f"EN: {translate(sentence)}\n")

Model loaded successfully!
DE: Der Morgen beginnt meistens sehr ruhig, wenn die ersten Sonnenstrahlen durch das Fenster scheinen. Viele Menschen trinken eine Tische Kaffee oder Tee, bevor sie sich auf den Weg zur Arbeit machen. Auf den Straßen herrscht oft schon große Hektik, während die Autos im dichten Verkehr stehen. Doch wenn man am Nachmittag einen Spaziergang im Park macht, kann man die frische Luft genießen und den Gesang der Vögel hören. Die Natur bietet uns einen wunderbaren Ausgleich zum stressigen Alltag in der modernen Stadt.


NameError: name 'greedy_decode' is not defined